# CMP AI LAB 교사용 직렬화 검증
학생과 동일한 학습 코드를 실행한 뒤 Python 모델과 웹앱 공통 JavaScript 추론을 32건 이상 비교합니다. 로컬 Node가 필요합니다. Colab 환경에 Node가 없다면 이 검증은 별도 개발 환경에서 실행하며 성공으로 표시하지 마세요.


In [ ]:
#@title ① 준비하기 · 코드 수정 없이 실행
"""CMP Colab / local training. No downloads, pickle loading, or external service needed."""
from pathlib import Path
import json, hashlib
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
import sklearn

FEATURES={'SIM':['pressure_kpa','rotation_rpm','pad_wear','slurry_ml_min'],
          'REAL':['pressure_scaled','rotation_scaled','pad_usage_scaled','slurry_scaled']}

def prepare(df):
    df=df.copy()
    required=['schema_version','source','wafer_id','run_id','stage','timestamp','mrr']
    missing=[c for c in required if c not in df]
    if missing:raise ValueError('필수 열이 없습니다: '+', '.join(missing))
    if not df['schema_version'].astype(str).eq('1.0.0').all():raise ValueError('자료 규격 버전은 1.0.0이어야 합니다.')
    sources=df['source'].dropna().unique().tolist()
    if len(sources)!=1 or sources[0] not in FEATURES:raise ValueError('SIM 또는 REAL 한 출처만 업로드하세요. 두 출처를 섞지 않습니다.')
    source=sources[0];features=FEATURES[source]
    stages=df['stage'].dropna().unique().tolist()
    if len(stages)!=1 or stages[0]!=('EDU' if source=='SIM' else 'A'):raise ValueError('SIM은 EDU, REAL은 A 단계 자료만 사용하세요.')
    if df[required].isna().any().any():raise ValueError('식별자·시간·정답에 빈칸이 있습니다.')
    if df['run_id'].astype(str).duplicated().any():raise ValueError('중복 run_id가 있습니다. 같은 자료를 다시 붙였는지 확인하세요.')
    for c in features+['mrr']:
        if c not in df:raise ValueError('필수 열이 없습니다: '+c)
        df[c]=pd.to_numeric(df[c],errors='coerce')
        if not np.isfinite(df[c]).all():raise ValueError(c+'의 빈칸·문자·무한값을 확인하세요. 자동으로 0을 넣지 않습니다.')
    if (df['mrr']<=0).any():raise ValueError('평균 제거율 정답은 양수여야 합니다. 출처와 계산을 확인하세요.')
    if source=='SIM':
        for c,low,high in [('duration_s',30,75),('initial_thickness_nm',1000,1000)]:
            if c not in df:raise ValueError('필수 열이 없습니다: '+c)
            values=pd.to_numeric(df[c],errors='coerce')
            if not values.between(low,high).all():raise ValueError(c+'의 교육용 범위를 확인하세요.')
        if not df.timestamp.astype(str).str.contains(r'(?:Z|[+-]\d{2}:\d{2})$',regex=True).all():raise ValueError('SIM 시각에 시간대가 필요합니다.')
    # REAL timestamp is a proprietary scaled ordering value, not a calendar date.
    if source=='REAL':df['_order']=pd.to_numeric(df['timestamp'],errors='coerce')
    else:df['_order']=pd.to_datetime(df['timestamp'],errors='coerce',utc=True)
    if df['_order'].isna().any():raise ValueError('시간/순서 값을 확인하세요.')
    groups=df.groupby('wafer_id',sort=False)['_order'].agg(['min','max']).sort_values(['min','max'],kind='stable')
    if len(groups)<30:raise ValueError('학습·검증·시험을 나누려면 서로 다른 웨이퍼가 30개 이상 필요합니다. 시뮬레이터의 실험표 60장을 수집하세요.')
    n=len(groups);a=n*70//100;b=n*85//100
    mapping={key:('train' if i<a else 'validation' if i<b else 'test') for i,key in enumerate(groups.index)}
    df['split']=df['wafer_id'].map(mapping)
    # A group that crosses a chronological boundary must not leak later records into training.
    for first,second in [('train','validation'),('validation','test')]:
        if df.loc[df.split==first,'_order'].max()>df.loc[df.split==second,'_order'].min():
            raise ValueError('동일 웨이퍼의 시간 구간이 분할 경계를 가로지릅니다. 웨이퍼별 한 행으로 요약하거나 경계를 다시 정하세요.')
    return df,source,features,stages[0]

def predict_json(model,frame):
    X=frame[model['features']].to_numpy(dtype=np.float32)
    output=[]
    for row in X:
        total=0.0
        for t in model['trees']:
            i=0;steps=0
            while t['left'][i]!=-1:
                steps+=1
                if steps>50:raise ValueError('이전 모델의 나무 구조가 잘못되었습니다.')
                i=t['left'][i] if row[t['feature'][i]]<=t['threshold'][i] else t['right'][i]
            total+=t['value'][i]
        output.append(total/len(model['trees']))
    return np.array(output)

def fit_and_export(frame,outdir,version='student-v1'):
    import re
    if not re.fullmatch(r'[A-Za-z0-9_-]{1,60}',str(version)):raise ValueError('모델 버전은 영문·숫자·밑줄·하이픈 1~60자로 적으세요.')
    df,source,features,stage=prepare(frame)
    train=df[df.split=='train'];val=df[df.split=='validation'];test=df[df.split=='test']
    rf=RandomForestRegressor(n_estimators=48,max_depth=8,min_samples_leaf=2,random_state=42,n_jobs=1)
    rf.fit(train[features],train.mrr)
    baseline=float(train.mrr.mean());metrics={}
    for name,data in [('validation',val),('test',test)]:
        p=rf.predict(data[features]);metrics[name]={'n':len(data),'mae':float(mean_absolute_error(data.mrr,p)),'baseline_mae':float(mean_absolute_error(data.mrr,np.full(len(data),baseline)))}
    model={'format':'daa-cmp-rf-v1','schema_version':'1.0.0','source':source,'stage':stage,'features':features,'target':'mrr','target_unit':'nm/min (educational)' if source=='SIM' else 'proprietary_scaled','inference_timing':'post_process_pre_inspection','model_version':str(version)[:80],'sklearn_version':sklearn.__version__,'training_ranges':{f:[float(train[f].min()),float(train[f].max())] for f in features},'split_counts':df.split.value_counts().to_dict(),'metrics':metrics,'split_rule':'wafer groups by time 70/15/15; training only; validation for policy; test once','trees':[]}
    from datetime import datetime,timezone
    model.update(model_id=source+'-'+version,source_type=source,feature_columns=features,feature_types={f:'number' for f in features},imputer_params={'strategy':'reject_missing','scaling':'none'},target_name='mrr',training_dataset_hash=hashlib.sha256(train.drop(columns=['_order']).to_csv(index=False).encode()).hexdigest(),trained_at=datetime.now(timezone.utc).isoformat(),model_format='daa-cmp-rf-v1',model_parameters={'n_estimators':48,'max_depth':8,'min_samples_leaf':2,'random_state':42},input_cast='float32')
    for est in rf.estimators_:
        t=est.tree_;model['trees'].append({'left':t.children_left.tolist(),'right':t.children_right.tolist(),'feature':t.feature.tolist(),'threshold':t.threshold.tolist(),'value':t.value[:,0,0].tolist()})
    all_pred=rf.predict(df[features]);delta=float(np.max(np.abs(all_pred-predict_json(model,df))))
    if delta>1e-9:raise RuntimeError('모델 저장 후 예측 검증에 실패했습니다.')
    model['export_max_abs_error']=delta
    out=Path(outdir);out.mkdir(parents=True,exist_ok=True)
    path=out/f'{source}_model_{version}.json';path.write_text(json.dumps(model,ensure_ascii=False,allow_nan=False,separators=(',',':')),encoding='utf-8')
    df['predicted_mrr']=all_pred;df['absolute_error']=abs(df.mrr-all_pred)
    if source=='SIM':
        df['predicted_thickness_nm']=pd.to_numeric(df.initial_thickness_nm)-all_pred*pd.to_numeric(df.duration_s)/60
    clean=df.drop(columns=['_order'])
    clean.to_csv(out/f'{source}_predictions_{version}.csv',index=False,encoding='utf-8-sig')
    clean[clean.split=='validation'].to_csv(out/f'{source}_validation_{version}.csv',index=False,encoding='utf-8-sig')
    clean[clean.split=='test'].to_csv(out/f'{source}_test_{version}.csv',index=False,encoding='utf-8-sig')
    (out/f'{source}_metrics_{version}.json').write_text(json.dumps(metrics,ensure_ascii=False,indent=2))
    return model,clean,metrics


import os, io, zipfile
try:
    from google.colab import files
    IN_COLAB=True
except ImportError:
    IN_COLAB=False
print('준비 완료. 다음 셀에서 CSV 한 개를 선택하세요.')
print('scikit-learn 버전:', sklearn.__version__)


## ② 데이터 선택
실험실에서 내려받은 CSV 하나를 선택합니다. 예비 실습은 `SIM_training.csv`, 공개 실측 비교는 `REAL_PHM2016_stage_A.csv`를 사용합니다. 판정 기록 CSV는 학습 원자료와 열 구성이 다릅니다.


In [ ]:
#@title ② CSV 한 개 불러오기
if IN_COLAB:
    uploaded=files.upload()
    if len(uploaded)!=1:raise ValueError('CSV 한 개만 선택하세요.')
    filename=next(iter(uploaded))
    if not filename.lower().endswith('.csv'):raise ValueError('CSV 파일이 필요합니다.')
    frame=pd.read_csv(io.BytesIO(uploaded[filename]))
else:
    filename=os.environ.get('CMP_DATA_PATH','SIM_my_experiments.csv')
    frame=pd.read_csv(filename)
print('파일:',Path(filename).name,' / 행:',len(frame))


## ③ 입력과 정답 구분
압력·회전·패드 사용·슬러리는 **입력**, `mrr`은 검사로 구한 **정답**입니다. 웨이퍼 ID, 순서, 검사 두께는 AI 입력에 넣지 않습니다.

웨이퍼를 시간순으로 정렬하여 학습 70%, 검증 15%, 시험 15%로 나눕니다. 장수는 정수로 나누므로 비율은 조금 달라질 수 있습니다. 같은 웨이퍼를 여러 묶음에 나누지 않습니다. 전처리를 정할 때에도 학습 자료만 사용합니다. 빈칸은 자동으로 0으로 채우지 않습니다.


In [ ]:
#@title ③ 자료와 분할 확인
prepared,source,features,stage=prepare(frame)
print('출처:',source,' / 단계:',stage)
print('입력 열:',', '.join(features))
print('정답 열: mrr')
print(prepared.groupby('split',sort=False).size().rename('웨이퍼 수').to_string())
print('\n자료 첫 3행')
display(prepared[['run_id']+features+['mrr','split']].head(3))
print('SIM 단위: 수업용 nm/min' if source=='SIM' else 'REAL 단위: 원공개자가 변환한 척도. nm/min으로 해석하지 않습니다.')


## ④ 기본 모델 하나 학습
여러 작은 결정나무의 예측을 평균하는 **랜덤 포레스트 회귀**를 사용합니다. 학생은 나무 수를 바꾸는 경쟁보다 자료와 결과를 해석합니다.

MAE는 예측과 정답 사이 거리의 평균입니다. 항상 학습 자료의 평균만 말하는 기준 모델과 비교하세요. AI가 기준 모델보다 나쁘면 그 결과도 기록합니다. 시험 자료를 보고 모델 설정을 다시 맞추지 않습니다.


In [ ]:
#@title ④ 학습하고 검증 오차 보기
MODEL_VERSION = "student-v1" #@param {type:"string"}
output_dir=Path('cmp_output')/MODEL_VERSION
model,predictions,metrics=fit_and_export(frame,output_dir,MODEL_VERSION)
v=metrics['validation']
print('검증 자료:',v['n'],'장')
print('AI 검증 MAE:',round(v['mae'],3))
print('평균 기준 모델 MAE:',round(v['baseline_mae'],3))
print('이 자료에서는 AI의 평균 오차가 더 작습니다.' if v['mae']<v['baseline_mae'] else '이 자료에서는 AI가 평균 기준 모델보다 낫지 않습니다. 자료의 변화·표본·입력 정보의 한계를 검토하세요.')
print('시험 결과는 검사 방침을 정한 뒤 다음 셀에서 확인합니다.')
display(predictions[predictions.split=='validation'][['run_id','mrr','predicted_mrr','absolute_error']].head(5))

def policy_table(data):
    results=[]
    for name,low,high in [('A',485,515),('B',490,510)]:
        counts={'방침':name,'통상 검사':0,'추가 검사':0,'보류':0,'통상 검사 중 실제 부적합':0}
        for _,row in data.iterrows():
            thick=float(row.predicted_thickness_nm)
            outside=any(float(row[f])<model['training_ranges'][f][0] or float(row[f])>model['training_ranges'][f][1] for f in features)
            status='보류' if thick<480 or thick>520 else ('추가 검사' if outside or thick<low or thick>high else '통상 검사')
            counts[status]+=1
            if status=='통상 검사' and not 480<=float(row.inspection_thickness_nm)<=520:counts['통상 검사 중 실제 부적합']+=1
        results.append(counts)
    return pd.DataFrame(results)

if source=='SIM':
    print('검증 자료 A/B 비교: 다음 셀 실행 전에 방침을 선택하세요.')
    display(policy_table(predictions[predictions.split=='validation']))


## ⑤ 검사 방침을 정하고 마지막 시험 확인
SIM 품질 규격은 **480~520 nm로 고정**합니다. A의 통상 검사 구간은 485~515, B는 490~510 nm입니다. 규격 안이지만 통상 구간 밖이면 추가 검사, 규격 밖이면 보류입니다. 학습 범위 밖 입력은 최소한 추가 확인 대상으로 봅니다. 통상 검사도 검사를 생략하지 않습니다.

먼저 ④ 실행 결과의 **검증 A/B 표**를 보고 활동지에 선택 이유를 씁니다. 방침을 바꿀 때는 검증 표만 근거로 사용합니다. 시험 자료는 최종 확인으로 해석하며, 결과가 마음에 들지 않는다고 재분할하지 않습니다. REAL에는 실제 검사 합격 규격이 없으므로 제거율 오차만 확인합니다.


In [ ]:
#@title ⑤ 검증 방침 비교와 최종 시험 · 결과 ZIP 저장
POLICY = "A" #@param ["A", "B"]
REASON = "활동지에 검증 표를 근거로 선택 이유를 기록합니다." #@param {type:"string"}
if source=='SIM':
    print('검증 자료 A/B 비교')
    print('④의 검증 표로 방침을 선택한 뒤 시험 결과를 확인합니다.')
    print('선택 방침:',POLICY,' / 근거:',REASON)
else:
    print('REAL은 품질 방침 대신 공개 제거율 척도의 오차를 비교합니다.')
t=metrics['test']
print('\n마지막 시험:',t['n'],'장 / AI MAE:',round(t['mae'],3),' / 평균 기준 MAE:',round(t['baseline_mae'],3))
print('이 시험 결과는 해당 자료의 결과이며, 실제 공장 적용 성능이나 학생 학습 효과를 뜻하지 않습니다.')
(output_dir/'my_decision.json').write_text(json.dumps({'source':source,'policy':POLICY if source=='SIM' else None,'reason':REASON,'model_version':MODEL_VERSION},ensure_ascii=False,indent=2),encoding='utf-8')
archive=Path(f'CMP_{source}_{MODEL_VERSION}_results.zip')
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for f in sorted(output_dir.iterdir()):z.write(f,f.name)
print('관제탑에 불러올 모델:',source+'_model_'+MODEL_VERSION+'.json')
print('학습 자료와 다른 새 실험 CSV를 관제탑에 접수하세요.')
if IN_COLAB:files.download(str(archive))
else:print('저장:',archive.resolve())


In [ ]:
CORE_SOURCE='(function(root,factory){if(typeof module===\'object\'&&module.exports)module.exports=factory();else root.CMP=factory();})(typeof window!==\'undefined\'?window:this,function(){\n\'use strict\';\nconst VERSION=\'1.0.0\';\nconst FEATURES={SIM:[\'pressure_kpa\',\'rotation_rpm\',\'pad_wear\',\'slurry_ml_min\'],REAL:[\'pressure_scaled\',\'rotation_scaled\',\'pad_usage_scaled\',\'slurry_scaled\']};\nconst COLUMNS=[\'schema_version\',\'source\',\'run_id\',\'wafer_id\',\'stage\',\'timestamp\',\'sequence\',\'pressure_set_kpa\',\'rotation_set_rpm\',\'pressure_kpa\',\'rotation_rpm\',\'pad_wear\',\'slurry_ml_min\',\'duration_s\',\'initial_thickness_nm\',\'mrr\',\'inspection_thickness_nm\',\'condition\',\'split\'];\nconst LABELS={ROUTINE:\'통상 검사\',REVIEW:\'추가 검사\',HOLD:\'보류 · 확인\',INVALID:\'판정 불가\',READY:\'시작 대기\',RUNNING:\'모의 운전\',STOP:\'안전 정지 유지\',CHECK:\'확인 필요\'};\nconst finite=v=>(typeof v===\'number\'||typeof v===\'string\'&&v.trim()!==\'\')&&Number.isFinite(Number(v));\nconst round=(x,n=4)=>Number(x.toFixed(n));\nfunction rng(seed){let a=seed>>>0;return()=>{a+=0x6D2B79F5;let t=a;t=Math.imul(t^t>>>15,t|1);t^=t+Math.imul(t^t>>>7,t|61);return((t^t>>>14)>>>0)/4294967296;};}\nfunction simulate(options={},seed=42){\n const o={pressure:30,rotation:60,duration:60,pad:.15,condition:\'standard\',sequence:1,prefix:\'SIM\',...options};\n for(const [k,a,b] of [[\'pressure\',20,36],[\'rotation\',40,70],[\'duration\',30,75],[\'pad\',0,1]])if(!finite(o[k])||o[k]<a||o[k]>b)throw Error(k+\' 입력 범위를 확인하세요.\');\n if(![\'standard\',\'changed\'].includes(o.condition))throw Error(\'알 수 없는 공정 조건입니다.\');\n const r=rng(seed),noise=()=>r()+r()+r()+r()+r()+r()-3;\n const trace=[];const slurry=round(200+noise()*2),p=round(o.pressure+noise()*.25),rpm=round(o.rotation+noise()*.45);\n const rel=1-.38*o.pad; const factor=o.condition===\'changed\'?.86:1;\n const mrr=round(Math.max(1,525*(p/30)*(rpm/60)*rel*(.9+.1*slurry/200)*factor+noise()*4));\n const thickness=round(1000-mrr*o.duration/60);\n for(let i=0;i<12;i++)trace.push({second:round(o.duration*i/11,1),pressure_kpa:round(p+noise()*.2,2),rotation_rpm:round(rpm+noise()*.35,2)});\n // Recenter traces so exported means equal the feature values used in the physics model.\n for(const [key,value] of [[\'pressure_kpa\',p],[\'rotation_rpm\',rpm]]){const avg=trace.reduce((s,v)=>s+v[key],0)/trace.length;trace.forEach(v=>v[key]=round(v[key]+value-avg,6));}\n const id=o.prefix+\'-\'+String(o.sequence).padStart(4,\'0\');\n const row={schema_version:VERSION,source:\'SIM\',run_id:id,wafer_id:id,stage:\'EDU\',timestamp:o.timestamp||new Date().toISOString(),sequence:o.sequence,pressure_set_kpa:o.pressure,rotation_set_rpm:o.rotation,pressure_kpa:p,rotation_rpm:rpm,pad_wear:o.pad,slurry_ml_min:slurry,duration_s:o.duration,initial_thickness_nm:1000,mrr,inspection_thickness_nm:thickness,condition:o.condition,split:\'\'};\n return {row,trace};\n}\nfunction csv(rows,columns){if(!rows.length)return (columns||COLUMNS).join(\',\')+\'\\r\\n\';const keys=columns||Object.keys(rows[0]);const esc=v=>{let s=v==null?\'\':String(v);if(/^[=+@]/.test(s)||/^-[^\\d.]/.test(s))s="\'"+s;return /[",\\r\\n]/.test(s)?\'"\'+s.replace(/"/g,\'""\')+\'"\':s;};return \'\\uFEFF\'+[keys,...rows.map(r=>keys.map(k=>r[k]))].map(a=>a.map(esc).join(\',\')).join(\'\\r\\n\');}\nfunction parseCSV(text){\n if(typeof text!==\'string\'||text.length>20e6)throw Error(\'CSV는 20 MB 이하로 준비하세요.\');\n text=text.replace(/^\\uFEFF/,\'\');let rows=[],row=[],v=\'\',quoted=false,ended=false;\n for(let i=0;i<text.length;i++){const c=text[i];if(quoted){if(c===\'"\'){if(text[i+1]===\'"\'){v+=\'"\';i++;}else {quoted=false;ended=true;}}else v+=c;}else if(c===\'"\'){if(v||ended)throw Error(\'CSV 따옴표 형식이 잘못되었습니다.\');quoted=true;}else if(c===\',\'){row.push(v);v=\'\';ended=false;}else if(c===\'\\n\'||c===\'\\r\'){if(c===\'\\r\'&&text[i+1]===\'\\n\')i++;row.push(v);if(row.some(x=>x!==\'\'))rows.push(row);row=[];v=\'\';ended=false;}else{if(ended&&c.trim())throw Error(\'CSV 따옴표 뒤의 문자를 확인하세요.\');v+=c;}}\n if(quoted)throw Error(\'CSV 따옴표가 닫히지 않았습니다.\');row.push(v);if(row.some(x=>x!==\'\'))rows.push(row);\n if(rows.length<2)throw Error(\'제목과 데이터 행이 있는 CSV를 선택하세요.\');const h=rows.shift().map(x=>x.trim());if(h.some(x=>!x)||new Set(h).size!==h.length)throw Error(\'빈 열 이름 또는 중복된 열 이름이 있습니다.\');if(rows.length>20000)throw Error(\'수업용 CSV는 20,000행 이하로 준비하세요.\');\n return rows.map((r,i)=>{if(r.length!==h.length)throw Error((i+2)+\'행의 열 개수가 다릅니다.\');const o=Object.create(null);h.forEach((k,j)=>{if([\'__proto__\',\'prototype\',\'constructor\'].includes(k))throw Error(\'허용되지 않는 열 이름입니다.\');o[k]=r[j].trim();});return o;});\n}\nfunction validateModel(m){\n if(!m||m.format!==\'daa-cmp-rf-v1\'||!FEATURES[m.source])throw Error(\'CMP Colab에서 저장한 모델 JSON이 필요합니다.\');\n if(typeof m.model_version!==\'string\'||!m.model_version||m.model_version.length>80||!m.split_counts||!Number.isInteger(m.split_counts.train)||m.split_counts.train<1||!m.metrics||!m.metrics.test||!Number.isFinite(m.metrics.test.mae))throw Error(\'모델 버전·학습 수·평가 정보가 없습니다.\');\n if(m.schema_version!==VERSION||m.stage!==(m.source===\'SIM\'?\'EDU\':\'A\'))throw Error(\'모델의 자료 규격 버전 또는 공정 단계가 맞지 않습니다.\');\n if(JSON.stringify(m.features)!==JSON.stringify(FEATURES[m.source]))throw Error(\'모델의 입력 열과 순서가 다릅니다.\');\n if(m.target!==\'mrr\'||m.inference_timing!==\'post_process_pre_inspection\')throw Error(\'예측 대상 또는 판정 시점이 맞지 않습니다.\');\n if(!Array.isArray(m.trees)||m.trees.length<1||m.trees.length>200)throw Error(\'모델의 나무 수가 허용 범위를 벗어납니다.\');\n let total=0;\n for(const t of m.trees){if(!t||!Array.isArray(t.left))throw Error(\'모델 구조가 잘못되었습니다.\');const n=t.left.length;total+=n;if(n<1||n>20000||total>200000)throw Error(\'모델이 너무 크거나 비어 있습니다.\');for(const k of [\'right\',\'feature\',\'threshold\',\'value\'])if(!Array.isArray(t[k])||t[k].length!==n)throw Error(\'모델 배열 길이가 다릅니다.\');let seen=new Set(),active=new Set();\n  function check(i,depth){if(!Number.isInteger(i)||i<0||i>=n||depth>50||active.has(i)||seen.has(i))throw Error(\'모델의 나무 구조가 올바르지 않습니다.\');active.add(i);seen.add(i);if(!Number.isFinite(t.value[i]))throw Error(\'모델 값이 유효하지 않습니다.\');if(t.left[i]===-1){if(t.right[i]!==-1)throw Error(\'모델 말단이 잘못되었습니다.\');}else{if(!Number.isInteger(t.feature[i])||t.feature[i]<0||t.feature[i]>=m.features.length||!Number.isFinite(t.threshold[i]))throw Error(\'모델 분기 값이 잘못되었습니다.\');check(t.left[i],depth+1);check(t.right[i],depth+1);}active.delete(i);}check(0,0);if(seen.size!==n)throw Error(\'연결되지 않은 모델 노드가 있습니다.\');}\n return m;\n}\nfunction predict(m,row){\n if(!m)throw Error(\'학습 모델을 먼저 불러오세요.\');if(row.schema_version!==VERSION)throw Error(\'데이터 규격 버전이 맞지 않습니다.\');if(row.source!==m.source)throw Error(\'데이터와 모델의 출처가 다릅니다. SIM과 REAL을 구분하세요.\');if(m.stage&&row.stage!==m.stage)throw Error(\'모델과 데이터의 공정 단계가 다릅니다.\');\n const x=m.features.map(k=>{if(!finite(row[k])||!Number.isFinite(Math.fround(Number(row[k]))))throw Error(k+\' 값이 비어 있거나 사용할 수 있는 숫자가 아닙니다.\');return Math.fround(Number(row[k]));});\n const predictions=m.trees.map(t=>{let i=0,steps=0;while(t.left[i]!==-1){if(++steps>50)throw Error(\'모델 탐색 오류\');i=x[t.feature[i]]<=t.threshold[i]?t.left[i]:t.right[i];}return t.value[i];});\n const value=predictions.reduce((s,v)=>s+v,0)/predictions.length;\n const outside=m.features.filter((k,i)=>m.training_ranges&&m.training_ranges[k]&&(Number(row[k])<m.training_ranges[k][0]||Number(row[k])>m.training_ranges[k][1]));\n return {value,outside};\n}\nfunction policy(thickness,kind=\'A\'){if(!Number.isFinite(thickness))return \'INVALID\';if(thickness<480||thickness>520)return \'HOLD\';const lo=kind===\'B\'?490:485,hi=kind===\'B\'?510:515;return thickness<lo||thickness>hi?\'REVIEW\':\'ROUTINE\';}\nfunction assess(m,row,kind=\'A\'){\n try{const p=predict(m,row);if(row.source===\'REAL\')return {status:\'REAL\',mrr:p.value,outside:p.outside,thickness:null,error:null};if(!finite(row.duration_s)||Number(row.duration_s)<30||Number(row.duration_s)>75||Number(row.initial_thickness_nm)!==1000)throw Error(\'수업용 연마 시간 30~75초, 시작 두께 1000 nm를 확인하세요.\');const thickness=Number(row.initial_thickness_nm)-p.value*Number(row.duration_s)/60;let status=policy(thickness,kind);if(p.outside.length&&status===\'ROUTINE\')status=\'REVIEW\';return {status,mrr:p.value,thickness,outside:p.outside,error:null};}catch(e){return {status:\'INVALID\',mrr:null,thickness:null,outside:[],error:e.message};}\n}\nfunction inspected(row){return row.source===\'SIM\'&&finite(row.inspection_thickness_nm)?(Number(row.inspection_thickness_nm)>=480&&Number(row.inspection_thickness_nm)<=520?\'PASS\':\'FAIL\'):null;}\nfunction dayKey(ts){const d=new Date(ts);return Number.isFinite(d.getTime())?new Intl.DateTimeFormat(\'en-CA\',{timeZone:\'Asia/Seoul\',year:\'numeric\',month:\'2-digit\',day:\'2-digit\'}).format(d):null;}\nfunction daily(rows,date){const selected=rows.filter(x=>x.row.source===\'SIM\'&&dayKey(x.row.timestamp)===date);const checked=selected.filter(x=>x.inspection_revealed&&inspected(x.row));const fail=checked.filter(x=>inspected(x.row)===\'FAIL\').length;return{total:selected.length,inspected:checked.length,uninspected:selected.length-checked.length,failed:fail,rate:checked.length?100*fail/checked.length:null};}\nfunction newSafety(){return{safe:true,latched:false,ack:false,permit:false,running:false,quality:\'ROUTINE\',dataOK:true};}\nfunction state(s){if(!s.safe||s.latched)return \'STOP\';if(!s.dataOK||[\'HOLD\',\'REVIEW\',\'INVALID\'].includes(s.quality))return \'CHECK\';return s.running?\'RUNNING\':\'READY\';}\nfunction safety(s,event){const n={...s};let message=\'\';switch(event){case\'ESTOP\':n.safe=false;n.latched=true;n.ack=false;n.permit=false;n.running=false;message=\'비상정지 발생. 정지 상태를 유지합니다.\';break;case\'RESTORE\':n.safe=true;message=\'위험 원인을 해소했습니다. RESET 전까지 정지 유지.\';break;case\'ACK\':n.ack=true;message=\'알람을 확인했습니다. 운전 상태는 바뀌지 않습니다.\';break;case\'RESET\':if(!n.safe){message=\'복구 거부: 위험 원인을 먼저 해소하세요.\';}else if(!n.latched){message=\'해제할 안전 정지 유지 상태가 없습니다.\';}else {n.latched=false;n.running=false;n.permit=false;message=\'복구 처리. 데이터·품질 조건을 다시 확인하며 자동 시작하지 않습니다.\';}break;case\'PERMIT\':if(state(n)===\'STOP\'||state(n)===\'CHECK\'){message=\'운전 허가 거부: 안전·데이터·품질 조건을 확인하세요.\';}else{n.permit=true;message=\'운전을 허가했습니다. 시작 버튼이 별도로 필요합니다.\';}break;case\'START\':if(state(n)===\'STOP\'||state(n)===\'CHECK\'||!n.permit){message=\'시작 거부: 복구·확인·운전 허가가 필요합니다.\';}else{n.running=true;message=\'별도 시작 요청으로 모의 운전을 시작했습니다.\';}break;case\'COMPLETE\':n.running=false;n.permit=false;message=\'모의 운전을 완료했습니다.\';break;default:throw Error(\'알 수 없는 안전 이벤트\');}return {next:n,message,state:state(n)};}\nfunction qualityUpdate(s,status){const n={...s,quality:status,dataOK:status!==\'INVALID\'};if([\'HOLD\',\'REVIEW\',\'INVALID\'].includes(status)){n.running=false;n.permit=false;}return n;}\nreturn{VERSION,FEATURES,COLUMNS,LABELS,finite,round,rng,simulate,csv,parseCSV,validateModel,predict,policy,assess,inspected,dayKey,daily,newSafety,state,safety,qualityUpdate};\n});\n'
#@title ⑥ Python과 관제탑 JavaScript의 실제 예측 비교
import shutil, subprocess
if not shutil.which('node'):raise RuntimeError('이 환경에 Node가 없습니다. 제공 소스의 npm test 또는 Node가 있는 개발 환경에서 검증하세요.')
sample=predictions[predictions.split=='test'].head(32).copy()
if len(sample)<20:sample=predictions.head(32).copy()
if len(sample)<20:raise ValueError('교차 비교에 20건 이상 필요합니다.')
Path('core.js').write_text(CORE_SOURCE)
Path('parity_model.json').write_text(json.dumps(model))
Path('parity_rows.json').write_text(sample.fillna('').to_json(orient='records'))
runner="const fs=require('fs'),C=require('./core.js');const m=C.validateModel(JSON.parse(fs.readFileSync('parity_model.json')));const rows=JSON.parse(fs.readFileSync('parity_rows.json'));console.log(JSON.stringify(rows.map(r=>C.predict(m,r).value)));"
Path('parity.js').write_text(runner)
js_pred=json.loads(subprocess.run(['node','parity.js'],check=True,capture_output=True,text=True).stdout)
py_pred=predict_json(model,sample)
delta=np.abs(np.array(js_pred)-py_pred)
if float(delta.max())>1e-6:raise AssertionError('직렬화 일치 검증 실패')
check=pd.DataFrame({'run_id':sample.run_id.to_numpy(),'python':py_pred,'javascript':js_pred,'abs_difference':delta})
check.to_csv('Python_vs_JavaScript.csv',index=False)
print('검증 건수:',len(check),'최대 절대 차이:',float(delta.max()),'허용: 1e-6')
display(check.head())
if IN_COLAB:files.download('Python_vs_JavaScript.csv')
